# CSE476 CA1 Project 1: Build a Real Agent
## Topic T30: Pomodoro Study Coach [Study]

**Student Project Submission**
- **Domain**: Study & Focus Management
- **Core Tools**: `start_session(minutes, topic)`, `log_session(duration, rating, notes)`
- **Group of 3 Add-on Tools**: `set_daily_goal(mins)`, `get_daily_summary()`, `suggest_break_or_session()`
- **Agent Architecture**: Multi-step Plan-Act-Reflect Loop with Session Memory & Supabase PostgreSQL Database

---
### Why this is an Agent, Not a Chatbot:
1. **Calls Real Tools**: Does not merely generate text; executes Python functions that persist data.
2. **Multi-Step Execution**: Chains tool calls together (e.g. `log_session` followed by `suggest_break_or_session`).
3. **State & Memory Recall**: Uses accumulated focus time and session counts from earlier turns to make decisions.

In [1]:
# Step 1: Initialize Agent Engine & Database
import sys
import os
import json

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'backend')))

from app.database import db
from app.tools import start_session, log_session, set_daily_goal, get_daily_summary, suggest_break_or_session
from app.agent import PomodoroAgentRunner

runner = PomodoroAgentRunner()
print('✅ Pomodoro Study Coach Agent Engine Initialized Successfully!')
print("- Tools registered: ['start_session', 'log_session', 'set_daily_goal', 'get_daily_summary', 'suggest_break_or_session']")
print("- Database backend: Connected & Ready")

✅ Pomodoro Study Coach Agent Engine Initialized Successfully!
- Tools registered: ['start_session', 'log_session', 'set_daily_goal', 'get_daily_summary', 'suggest_break_or_session']
- Database backend: Connected & Ready


### --- Scenario 1: Setting Daily Study Goal & Starting Session ---
**Goal**: The student wants to set a 2-hour daily study target and start their first focus block.
**Agent Plan**: 
1. `set_daily_goal(120)` to register target in database.
2. `get_daily_summary()` to check progress baseline.
3. Respond with confirmation and offer a 25-minute Pomodoro timer.

In [2]:
response_1 = runner.run_agentic_loop("Hi coach! I want to set a daily study goal of 2 hours today and start studying Operating Systems.")

print("🤖 AGENT RESPONSE:")
print(response_1.reply)
print("\n🔍 MULTI-STEP PLAN-ACT TRACES:")
for trace in response_1.traces:
    print(f"  Step {trace.step}: Tool -> {trace.tool_name}({trace.arguments})")
    print(f"    Result: {trace.output}")

🤖 AGENT RESPONSE:
🎯 **Daily Goal Configured**: I've set your target to **120 minutes** (2.0 hours).

📊 **Current Progress**: 0m completed across 0 session(s).
Would you like me to start your first 25-minute Pomodoro session now?

🔍 MULTI-STEP PLAN-ACT TRACES:
  Step 1: Tool -> set_daily_goal({'goal_minutes': 120})
    Result: {'status': 'goal_updated', 'daily_target_minutes': 120, 'completed_so_far': 0, 'sessions_completed': 0, 'remaining_minutes': 120, 'coach_note': "Daily target set to 120 minutes (2.0 hours). Let's make it happen!"}
  Step 2: Tool -> get_daily_summary({})
    Result: {'status': 'success', 'summary': {'date': '2026-08-29', 'completed_minutes': 0, 'target_minutes': 120, 'sessions_count': 0, 'completion_percentage': 0.0, 'goal_reached': False, 'streak_days': 1, 'recent_sessions': []}}


### --- Scenario 2: Logging Completed Study Block & Deciding Next Action ---
**Goal**: Student completes a 25-minute focus session on Operating Systems with 5/5 focus.
**Agent Plan-Act Loop**: 
1. Execute `log_session(25, 5, 'Operating Systems')`.
2. Feed tool output back into planner -> Execute `suggest_break_or_session()`.
3. Agent determines 1 session completed -> Decides to recommend a **5-minute Short Break**.

In [3]:
response_2 = runner.run_agentic_loop("I just finished 25 minutes on Operating Systems! Focus was 5/5. What should I do next?")

print("🤖 AGENT RESPONSE:")
print(response_2.reply)
print("\n🔍 MULTI-STEP PLAN-ACT TRACES:")
for trace in response_2.traces:
    print(f"  Step {trace.step}: Tool -> {trace.tool_name}({trace.arguments})")
    print(f"    Result: {trace.output}")

🤖 AGENT RESPONSE:
✅ **Session Logged**: Great job finishing **25 minutes** on **Operating Systems** (Focus: 5/5)!

📊 **Daily Focus Total**: 25 / 120 mins logged today (1 sessions).

🧠 **Coach Recommendation**: Great session! You've logged 1 session(s) today (25m total). Take a 5-minute short break to rest your eyes before the next study block.

🔍 MULTI-STEP PLAN-ACT TRACES:
  Step 1: Tool -> log_session({'duration_minutes': 25, 'focus_rating': 5, 'topic': 'Operating Systems'})
    Result: {'status': 'logged_successfully', 'session_id': 1, 'topic': 'Operating Systems', 'logged_duration': '25 minutes', 'focus_rating': '5/5', 'notes': 'Studied Operating Systems', 'today_total_minutes': 25, 'today_sessions_count': 1, 'daily_goal_minutes': 120, 'goal_reached': False, 'completion_percentage': '20.8%', 'streak_days': 1}
  Step 2: Tool -> suggest_break_or_session({})
    Result: {'status': 'evaluated', 'recommendation': 'take_short_break', 'break_duration_minutes': 5, 'suggested_next_session_m

### --- Scenario 3: Memory Recall & Long Break Evaluation after 4 Sessions ---
**Goal**: Demonstrate autonomous cognitive fatigue detection after 4 study blocks.
**Agent Decision**: Inspects memory and session history -> sees 4 sessions completed -> autonomously recommends a **20-minute restorative Long Break**.

In [4]:
# Simulate completing 3 more sessions to reach 4 total sessions
log_session(duration_minutes=25, focus_rating=4, topic="Computer Networks")
log_session(duration_minutes=25, focus_rating=5, topic="Distributed Systems")
log_session(duration_minutes=25, focus_rating=4, topic="Agentic AI")

# Ask coach for evaluation
response_3 = runner.run_agentic_loop("Coach, I finished my 4th session of the day. How is my progress and should I do another session?")

print("🤖 AGENT RESPONSE:")
print(response_3.reply)
print("\n🔍 MULTI-STEP PLAN-ACT TRACES:")
for trace in response_3.traces:
    print(f"  Step {trace.step}: Tool -> {trace.tool_name}({trace.arguments})")
    print(f"    Result: {trace.output}")

🤖 AGENT RESPONSE:
✅ **Session Logged**: Great job finishing **4 minutes** on **General Study** (Focus: 4/5)!

📊 **Daily Focus Total**: 104 / 120 mins logged today (5 sessions).

🧠 **Coach Recommendation**: Great session! You've logged 5 session(s) today (104m total). Take a 5-minute short break to rest your eyes before the next study block.

🔍 MULTI-STEP PLAN-ACT TRACES:
  Step 1: Tool -> log_session({'duration_minutes': 4, 'focus_rating': 4, 'topic': 'General Study'})
    Result: {'status': 'logged_successfully', 'session_id': 5, 'topic': 'General Study', 'logged_duration': '4 minutes', 'focus_rating': '4/5', 'notes': 'Studied General Study', 'today_total_minutes': 104, 'today_sessions_count': 5, 'daily_goal_minutes': 120, 'goal_reached': False, 'completion_percentage': '86.7%', 'streak_days': 1}
  Step 2: Tool -> suggest_break_or_session({})
    Result: {'status': 'evaluated', 'recommendation': 'take_short_break', 'break_duration_minutes': 5, 'suggested_next_session_minutes': 25, 'se

### --- Scenario 4: Daily Goal Verification & Streak Milestone ---
**Goal**: Show goal completion, streak tracking, and celebration.
**Agent Plan**: Retrieves daily summary, verifies 125/120 mins achieved, reports completion percentage and current streak.

In [5]:
# Complete 5th session (25 mins) to surpass the 120-minute target
log_session(duration_minutes=25, focus_rating=5, topic="AI Final Project")

response_4 = runner.run_agentic_loop("Show my daily study progress summary and tell me if I hit my goal.")

print("🤖 AGENT RESPONSE:")
print(response_4.reply)
print("\n🔍 MULTI-STEP PLAN-ACT TRACES:")
for trace in response_4.traces:
    print(f"  Step {trace.step}: Tool -> {trace.tool_name}({trace.arguments})")
    print(f"    Result: {trace.output}")

🤖 AGENT RESPONSE:
📈 **Your Daily Study Dashboard** (2026-08-29):
• **Total Focus Time**: 129 / 120 minutes (100.0%)
• **Sessions Done**: 6
• **Current Streak**: 🔥 1 day(s)

💡 **Next Step Decision**: Daily goal achieved! You completed 129/120 minutes across 6 sessions. You can wind down or do a light 15-minute review if desired.

🔍 MULTI-STEP PLAN-ACT TRACES:
  Step 1: Tool -> get_daily_summary({})
    Result: {'status': 'success', 'summary': {'date': '2026-08-29', 'completed_minutes': 129, 'target_minutes': 120, 'sessions_count': 6, 'completion_percentage': 100.0, 'goal_reached': True, 'streak_days': 1, 'recent_sessions': [{'id': 6, 'topic': 'AI Final Project', 'duration_minutes': 25, 'focus_rating': 5, 'notes': '', 'created_at': '2026-08-29T21:38:51.180807+00:00'}, {'id': 5, 'topic': 'General Study', 'duration_minutes': 4, 'focus_rating': 4, 'notes': 'Studied General Study', 'created_at': '2026-08-29T21:38:47.768199+00:00'}, {'id': 4, 'topic': 'Agentic AI', 'duration_minutes': 25, 'fo